# 03 — Model: LightGBM

Requires `train_features.csv` / `test_features.csv` from **01_feature_engineering.ipynb**.

Run `!pip install lightgbm` once if you don't already have it.

Native categorical support (pass `categorical_feature`). Tune `num_leaves` / `learning_rate` /
`n_estimators` further if you have spare compute — this is a reasonable starting config, not exhaustively
tuned. Saves `oof_lgb.csv` and `test_pred_lgb.csv` for the ensembling notebook.

In [ ]:
!pip install -q lightgbm


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DATA_DIR = "."   # <-- folder with train_features.csv / test_features.csv from notebook 01
N_FOLDS = 5
SEED = 42

train_fe = pd.read_csv(f"{DATA_DIR}/train_features.csv")
test_fe = pd.read_csv(f"{DATA_DIR}/test_features.csv")

num_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours',
            'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
            'notif_per_hour', 'app_per_hour', 'mins_per_appopen', 'mins_per_notif', 'productivity',
            'sleep_screen_sum', 'nonscreen_hours', 'screen_plus_weekend', 'screen_sleep_ratio',
            'sm_ratio', 'game_ratio', 'work_ratio', 'screen_minus_work', 'weekday_weekend_ratio',
            'screen_x_sm', 'screen_x_weekend', 'sm_x_weekend', 'screen_x_sleep']
cat_cols = ['gender', 'stress_level', 'academic_work_impact']
feat_cols = num_cols + cat_cols

X = train_fe[feat_cols].copy()
Xtest = test_fe[feat_cols].copy()
y = train_fe['addicted_label'].values

for c in cat_cols:
    X[c] = X[c].astype('category')
    Xtest[c] = Xtest[c].astype('category')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(X.shape, Xtest.shape)

import lightgbm as lgb
import time


In [ ]:
oof_lgb = np.zeros(len(X))
test_lgb = np.zeros(len(Xtest))

lgb_params = dict(
    objective='binary', metric='auc', learning_rate=0.03, num_leaves=255,
    min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=2.0, n_estimators=3000, random_state=SEED, verbosity=-1
)

t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(folds):
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X.iloc[tr_idx], y[tr_idx],
        eval_set=[(X.iloc[va_idx], y[va_idx])],
        categorical_feature=cat_cols,
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    p_va = model.predict_proba(X.iloc[va_idx])[:, 1]
    oof_lgb[va_idx] = p_va
    test_lgb += model.predict_proba(Xtest)[:, 1] / N_FOLDS
    print(f"fold {fold} auc={roc_auc_score(y[va_idx], p_va):.5f}  best_iter={model.best_iteration_}  ({time.time()-t0:.0f}s elapsed)")

print("LightGBM OOF AUC:", roc_auc_score(y, oof_lgb))


In [ ]:
pd.DataFrame({'id': train_fe['id'], 'oof_pred': oof_lgb}).to_csv(f"{DATA_DIR}/oof_lgb.csv", index=False)
pd.DataFrame({'id': test_fe['id'], 'test_pred': test_lgb}).to_csv(f"{DATA_DIR}/test_pred_lgb.csv", index=False)
print("saved oof_lgb.csv and test_pred_lgb.csv")
